In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 路径

# =========================
PROJECT_ROOT = Path("../..").resolve()

csv_1710 = PROJECT_ROOT /"results"/"final"/"Probe_compare_all_ckpt_summary__STP1710.7_vs_control.csv"

csv_1717 = PROJECT_ROOT /"results"/"final"/"Probe_compare_all_ckpt_summary__STP1717.1_vs_control.csv"

df_1710 = pd.read_csv(csv_1710)
df_1717 = pd.read_csv(csv_1717)

# 防止列名有隐藏空格
df_1710.columns = df_1710.columns.str.strip()
df_1717.columns = df_1717.columns.str.strip()

# =========================
# 统计函数：mean (95% CI using 1.96 * SE)
# =========================
def mean_ci_str(x):
    x = pd.Series(x).dropna().astype(float)
    n = len(x)
    if n == 0:
        return "NA"
    m = x.mean()
    sd = x.std(ddof=1) if n > 1 else 0.0
    se = sd / np.sqrt(n) if n > 1 else 0.0
    lo = max(0, m - 1.96 * se)
    hi = min(1, m + 1.96 * se)
    return f"{m:.3f} ({lo:.3f}–{hi:.3f})"

# =========================
# 构建表格
# =========================
def build_table(df, task_name):
    rows = []

    configs = [
        ("LDA", "In-sample", "AUROC_LDA_insample", "AUPRC_LDA_insample", "ACC_LDA_insample"),
        ("Logistic Regression", "In-sample", "AUROC_LR_insample", "AUPRC_LR_insample", "ACC_LR_insample"),
        ("LDA", "LOGO", "AUROC_LDA_logo", "AUPRC_LDA_logo", "ACC_LDA_logo"),
        ("Logistic Regression", "LOGO", "AUROC_LR_logo", "AUPRC_LR_logo", "ACC_LR_logo"),
    ]

    for model, setting, c1, c2, c3 in configs:
        missing = [c for c in [c1, c2, c3] if c not in df.columns]
        if missing:
            raise ValueError(f"{task_name} 缺少列: {missing}")

        rows.append({
            "Task": task_name,
            "Model": model,
            "Setting": setting,
            "AUROC": mean_ci_str(df[c1]),
            "AUPRC": mean_ci_str(df[c2]),
            "ACC": mean_ci_str(df[c3]),
        })

    return rows

# =========================
# 拼接两个任务
# =========================
rows = []
rows += build_table(df_1710, "STP1710.7 vs control")
rows += build_table(df_1717, "STP1717.1 vs control")

table_df = pd.DataFrame(rows)

# =========================
# 展示版：重复 task 留空
# =========================
table_display = table_df.copy()
for i in range(1, len(table_display)):
    if table_display.loc[i, "Task"] == table_display.loc[i - 1, "Task"]:
        table_display.loc[i, "Task"] = ""

# =========================
# 输出
# =========================
print("\n=== FINAL TABLE ===\n")
print(table_display.to_string(index=False))

# =========================
# 保存
#OUTPUT_DIR = PROJECT_ROOT /"results"/"runs"/"Probe_compare"/"Task1717"/"logs1"
# =========================
out_dir = PROJECT_ROOT / "figures"/"qc"/"log1"
out_dir.mkdir(parents=True, exist_ok=True)

csv_out = out_dir / "probe_summary_table.csv"


table_df.to_csv(csv_out, index=False)

# with open(txt_out, "w") as f:
#     f.write(table_display.to_string(index=False))

# with open(latex_out, "w") as f:
#     f.write(table_display.to_latex(index=False, escape=False))

print(f"\nSaved CSV: {csv_out}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# =========================
# 路径
# =========================
PROJECT_ROOT = Path("../..").resolve()

csv_1710 = PROJECT_ROOT /"results"/"final"/"Probe_compare_all_ckpt_summary__STP1710.7_vs_control.csv"

csv_1717 = PROJECT_ROOT /"results"/"final"/"Probe_compare_all_ckpt_summary__STP1717.1_vs_control.csv"

df_1710 = pd.read_csv(csv_1710)
df_1717 = pd.read_csv(csv_1717)

# 防止列名有空格
df_1710.columns = df_1710.columns.str.strip()
df_1717.columns = df_1717.columns.str.strip()

# =========================
# 从宽表整理成长表（只取 LOGO + AUROC）
# =========================
plot_1710 = pd.DataFrame({
    "task": ["STP1710.7 vs control"] * (len(df_1710) * 2),
    "model_short": ["LDA"] * len(df_1710) + ["LogReg"] * len(df_1710),
    "AUROC": list(df_1710["AUROC_LDA_logo"].values) + list(df_1710["AUROC_LR_logo"].values),
    "seed": list(df_1710["seed"].values) + list(df_1710["seed"].values),
})

plot_1717 = pd.DataFrame({
    "task": ["STP1717.1 vs control"] * (len(df_1717) * 2),
    "model_short": ["LDA"] * len(df_1717) + ["LogReg"] * len(df_1717),
    "AUROC": list(df_1717["AUROC_LDA_logo"].values) + list(df_1717["AUROC_LR_logo"].values),
    "seed": list(df_1717["seed"].values) + list(df_1717["seed"].values),
})

df_plot = pd.concat([plot_1710, plot_1717], ignore_index=True)
df_plot = df_plot.dropna(subset=["AUROC"]).copy()

task_order = ["STP1710.7 vs control", "STP1717.1 vs control"]
task_labels = ["STP1710.7 vs control", "STP1717.1 vs control"]
hue_order = ["LDA", "LogReg"]

df_plot["task"] = pd.Categorical(df_plot["task"], categories=task_order, ordered=True)

print(df_plot.head())
print(df_plot.groupby(["task", "model_short"]).size())

# =========================
# 统一 theme（尽量贴近你现有 AUROC / ΔAUROC 图）
# =========================
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 13,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "legend.fontsize": 10.5,
    "axes.linewidth": 1.0,
    "grid.linewidth": 0.7,
    "figure.dpi": 300,
    "savefig.dpi": 300,
})

# 与你其余结果图统一：蓝色主模型 + 灰色对照
palette = {
    "LDA": "#B5BBC0",     # 中性灰
    "LogReg": "#4F86B9"   # 主蓝色
}

fig, ax = plt.subplots(figsize=(7.2, 4.8))

# =========================
# 箱线图
# =========================
sns.boxplot(
    data=df_plot,
    x="task",
    y="AUROC",
    hue="model_short",
    order=task_order,
    hue_order=hue_order,
    palette=palette,
    width=0.55,
    fliersize=0,
    linewidth=1.3,
    saturation=0.95,
    boxprops=dict(alpha=0.95),
    whiskerprops=dict(linewidth=1.2, color="#444444"),
    capprops=dict(linewidth=1.2, color="#444444"),
    medianprops=dict(color="black", linewidth=2.0),
    ax=ax
)

# =========================
# 原始点
# =========================
sns.stripplot(
    data=df_plot,
    x="task",
    y="AUROC",
    hue="model_short",
    order=task_order,
    hue_order=hue_order,
    palette=palette,
    dodge=True,
    size=8,
    alpha=0.7,
    edgecolor="#555555",
    linewidth=0.6,
    jitter=0.08,
    ax=ax
)

# =========================
# 去重 legend
# =========================
handles, labels = ax.get_legend_handles_labels()
seen = set()
new_handles, new_labels = [], []
for h, l in zip(handles, labels):
    if l not in seen:
        seen.add(l)
        new_handles.append(h)
        new_labels.append(l)

ax.legend(
    new_handles[:2],
    new_labels[:2],
    title="",
    frameon=False,
    loc="lower right"
)

# =========================
# mean diamond
# =========================
offset_map = {
    "LDA": -0.20,
    "LogReg": 0.20,
}

for i, task in enumerate(task_order):
    for model in hue_order:
        sub = df_plot[(df_plot["task"] == task) & (df_plot["model_short"] == model)]
        if len(sub) == 0:
            continue
        y_mean = sub["AUROC"].mean()
        x_mean = i + offset_map[model]
        ax.scatter(
            x_mean, y_mean,
            marker="D",
            s=72,
            color="white",
            edgecolor="#222222",
            linewidth=1.2,
            zorder=7
        )

# =========================
# 坐标与样式
# =========================
TITLE_SIZE  = 12
LABEL_SIZE  = 12
TICK_SIZE   = 10
LEGEND_SIZE = 10


ax.set_xlabel("")
ax.set_ylabel("AUROC", fontsize=LABEL_SIZE)
ax.set_title("Probe comparison under frog-level LOFO", fontsize=TITLE_SIZE, pad=10)

ax.set_xticklabels(task_labels)
ax.set_ylim(0.15, 0.95)

ax.grid(axis="y", linestyle="--", alpha=0.18, color="#D5D5D5")
ax.grid(axis="x", visible=False)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color("#666666")
ax.spines["bottom"].set_color("#666666")

ax.margins(x=0.05)

plt.subplots_adjust(left=0.12, right=0.97, top=0.90, bottom=0.16)

#out_dir = base_dir / "Figures"
out_dir = PROJECT_ROOT / "figures"/"qc"/"log1"
out_dir.mkdir(parents=True, exist_ok=True)
png_out = out_dir / "LR_LDA.png"

plt.savefig(png_out, dpi=300, bbox_inches="tight")

plt.show()

